# PFAS Groundwater — Figure phare (mémoire) : HGT → Fusion → Stacking + Optuna HPO (T1a, v2)

**Tâche T1a** : prédire un dépassement réglementaire EPA-2024 (binaire) à partir des  
features de contexte seulement (mode prédictif strict — zéro mesure PFAS, zéro lat/lon, C-LOC.1).

**Ce notebook est AUTONOME** (CLAUDE.md §4) : il bootstrappe `src/` et le parquet versionné  
via `git clone` (zéro Google Drive, zéro gdown), installe PyTorch Geometric pour la  
roue torch/CUDA du runtime, puis orchestre les quatre études Optuna séparées + le run final  
avec les meilleurs hyperparamètres + figures SHAP + figure phare du mémoire.

**Quatre études Optuna indépendantes (OOF spatial, jamais sur test) :**
1. HGT seul — espace : hidden, layers, dropout, heads, lr, weight_decay, k_spatial, cap_km_spatial.
2. XGBoost tabulaire — espace : n_estimators, max_depth, lr, subsample, colsample, reg_lambda, min_child_weight.
3. RandomForest tabulaire — espace : n_estimators, max_depth, max_features, min_samples_leaf, class_weight.
4. Stacking méta-apprenant — espace : max_depth, n_estimators, learning_rate du méta-XGB.

**Anti-fuite (CLAUDE.md §3.1/§3.2) :**
- Aucune mesure PFAS ni colonne dérivée en feature (LEAKAGE_BLOCKLIST respectée).
- HPO uniquement sur AUC OOF spatiale (per-fold mean sur les k blocs LOBO).
- Validation croisée spatiale par blocs K-Means k=8 ; bras random pour mesurer l'inflation Δ.
- Bords cross-blocs coupés et assertés = 0 par relation (C-SPAT.2/5).

**Courbes d'entraînement (CLAUDE.md §3.8) :** perte et AUC train+val par pli pour le HGT ;  
courbe boosting XGB (AUC vs n_estimators). Figures dans `experiments/hgt_fusion_stacking_t1_v2/figures/`.

**Runtime recommandé :** T4 GPU (Runtime > Change runtime type > T4 GPU).  
Pour `SMOKE_TEST=True`, un runtime CPU High-RAM suffit.

---
**IMPORTANT avant Colab :** pousser le code sur le remote avant d'ouvrir ce notebook :
```bash
git add src/hgt_fusion_stacking_t1_optuna.py notebooks/hgt_fusion_stacking_t1_v2_optuna_colab.ipynb
git commit -m 'feat(optuna): HPO studies + figures SHAP for figure phare v2'
git push
```
Puis mettre à jour `GIT_REF` ci-dessous (branche ou SHA de commit).

## Cell 0 — Paramètres utilisateur (à ajuster avant de lancer)

In [ ]:
# ============================================================
# PARAMÈTRES UTILISATEUR — ajuster avant de lancer
# ============================================================

SMOKE_TEST = True  # True = sanity check CPU < ~3 min ; False = run complet GPU

# Repo à cloner (code + dataset versionnés ensemble, zéro Drive)
REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF  = "main"  # branche ou SHA de commit ; doit contenir src/hgt_fusion_stacking_t1_optuna.py

# Dataset versionné dans le repo — arrive AVEC le clone, pas de gdown
DATA_PATH = "data/CA-PFAS-ASGWS_v2.parquet"  # relatif à la racine du repo

# Répertoire d'expérience (artefacts écrits ICI, dans l'espace de travail)
EXP_DIR       = "experiments/hgt_fusion_stacking_t1_v2"          # run complet
SMOKE_EXP_DIR = "experiments/hgt_fusion_stacking_t1_v2/_smoke"   # smoke

# Nombre de trials Optuna par étude (réduit en smoke)
OPTUNA_TRIALS_SMOKE = 3
OPTUNA_TRIALS_FULL  = 30  # 4 × 30 = 120 trials total

# Hyperparamètres du run FINAL (après HPO, utilisés si on relance avec best_params)
# Defaults du module (overridés par les best_params Optuna dans les cellules plus bas)
DEFAULT_HIDDEN       = 64
DEFAULT_LAYERS       = 2
DEFAULT_DROPOUT      = 0.3
DEFAULT_HEADS        = 4
DEFAULT_LR           = 5e-3
DEFAULT_WEIGHT_DECAY = 5e-4
DEFAULT_K_SPATIAL    = 8
DEFAULT_CAP_KM       = 1.5
DEFAULT_MAX_EPOCHS   = 400
DEFAULT_PATIENCE     = 50
COMPUTE_DELTA        = True   # bras random pour mesurer inflation Δ
SEED = 42

# ============================================================
# ESTIMATION DE DURÉE (SMOKE_TEST=False, Colab T4 GPU)
# ============================================================
# Smoke (CPU, 500 puits, 3 blocs) :
#   4 études Optuna (3 trials chacune) : ~70-80s
#   Run final (3 blocs, 15 époques)    : ~50-60s
#   Figures + SHAP                     : ~20s
#   TOTAL SMOKE                        : ~3 min
#
# Run complet (T4 GPU, 11333 puits, 8 blocs) :
#   4 études Optuna (30 trials)        : ~2-3 h
#     Study 1 HGT (30 × backbone)      : ~1.5-2 h  [dominant]
#     Study 2 XGB (30 × LOBO tabular)  : ~15-20 min
#     Study 3 RF  (30 × LOBO tabular)  : ~10-15 min
#     Study 4 Stacking (backbone×1     :
#             + 30 × meta LOBO)        : ~20-30 min
#   Run final avec best_params (1 backbone complet) : ~60-90 min
#   Figures + SHAP                     : ~10-15 min
#   TOTAL FULL (avec COMPUTE_DELTA)    : ~4-5 h
#   TOTAL FULL (sans COMPUTE_DELTA)    : ~3-4 h
#
# Note : la study HGT (30 backbones = 30 × LOBO-8) est le goulot d'étranglement.
# Pour un premier run, on peut réduire OPTUNA_TRIALS_FULL à 15 (→ ~2.5-3 h total).
# ============================================================

print("Paramètres configurés.")
print(f"  SMOKE_TEST        : {SMOKE_TEST}")
print(f"  REPO_URL          : {REPO_URL}")
print(f"  GIT_REF           : {GIT_REF}")
print(f"  DATA_PATH         : {DATA_PATH}")
print(f"  EXP_DIR           : {EXP_DIR}")
print(f"  OPTUNA_TRIALS     : {'SMOKE='+str(OPTUNA_TRIALS_SMOKE) if SMOKE_TEST else 'FULL='+str(OPTUNA_TRIALS_FULL)} (par étude)")
print(f"  COMPUTE_DELTA     : {COMPUTE_DELTA}")
if not SMOKE_TEST:
    print()
    print("  Estimation run complet (Colab T4 GPU, COMPUTE_DELTA=True) : ~4-5 h")
    print("  Pour un run rapide : réduire OPTUNA_TRIALS_FULL=15 → ~2.5-3 h")

## Cell 1 — Détection GPU + versions Python/torch/CUDA

In [ ]:
import sys
import platform

print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB : {IN_COLAB}")

try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"torch    : {torch.__version__}  CUDA avail: {cuda_ok}")
    if cuda_ok:
        print(f"GPU      : {torch.cuda.get_device_name(0)}")
        print(f"CUDA     : {torch.version.cuda}")
    else:
        print("AVERTISSEMENT : pas de GPU détecté.")
        if SMOKE_TEST:
            print("  SMOKE_TEST=True -> run CPU attendu et correct.")
        else:
            print("  SMOKE_TEST=False -> le backbone HGT a besoin du GPU (~4-5 h sur T4).")
            print("  Aller dans : Runtime > Change runtime type > T4 GPU")
except ImportError:
    print("torch non encore installé (sera disponible après la Cell 2).")

print()
print("Note : XGBoost, RF et le méta-apprenant Stacking utilisent le CPU.")
print("Note : seul le backbone HGT bénéficie du GPU (PyTorch Geometric).")

## Cell 2 — Installation des dépendances épinglées + vérification des imports

PyG compilé doit correspondre à `torch.__version__` + tag CUDA.  
On détecte les deux au runtime et on installe depuis l'index de roues correspondant.  
Cette cellule est idempotente : si PyG est déjà importable elle saute la lourde installation.

In [ ]:
import subprocess
import sys

# --- Dépendances de base (hors PyG) ---
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyarrow>=14.0",
    "pyyaml>=6.0",
    "scikit-learn>=1.4",
    "matplotlib>=3.7",
    "xgboost>=2.0",
    "lightgbm>=4.0",
    "optuna>=3.0",
    "shap>=0.44",
], check=True)

# --- PyTorch Geometric (doit correspondre au runtime torch + CUDA) ---
def ensure_pyg():
    try:
        import torch_geometric
        print(f"torch_geometric déjà présent : {torch_geometric.__version__}")
        return
    except ImportError:
        pass
    import torch
    tv = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    tag = f"cu{cuda.replace('.', '')}" if (cuda and torch.cuda.is_available()) else "cpu"
    idx = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
    print(f"Installation de torch_geometric pour torch={tv}, device={tag}")
    print(f"  Index de roues : {idx}")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "torch_geometric", "-f", idx
    ], check=True)
    # Extensions compilées optionnelles (scatter/sparse) — ignore l'échec sur CPU
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "torch_scatter", "torch_sparse", "-f", idx
    ])

ensure_pyg()

# --- Vérification des imports PyG requis ---
import torch_geometric
print(f"PyG : {torch_geometric.__version__}")

from torch_geometric.utils import dropout_edge
print("  dropout_edge       OK")
from torch_geometric.nn import GraphNorm
print("  GraphNorm          OK")
from torch_geometric.nn import SAGEConv
print("  SAGEConv           OK")
from torch_geometric.nn import HeteroConv
print("  HeteroConv         OK")
from torch_geometric.nn import HGTConv
print("  HGTConv            OK")

# --- Vérification des autres imports ---
import xgboost as xgb
print(f"xgboost  : {xgb.__version__}")
import lightgbm as lgb
print(f"lightgbm : {lgb.__version__}")
import optuna
print(f"optuna   : {optuna.__version__}")
import shap
print(f"shap     : {shap.__version__}")

print()
print("Toutes les dépendances satisfaites.")

## Cell 3 — Clone du repo (ramène `src/` ET `data/` — zéro Drive, zéro gdown)

Le parquet `data/CA-PFAS-ASGWS_v2.parquet` est **versionné dans le repo** et arrive  
automatiquement avec le clone. Cette cellule est idempotente : si le répertoire existe déjà  
elle vérifie juste le `GIT_REF`.

**Garde-fou anti-code-obsolète :** après le bootstrap, vérifie que les symboles requis du  
module Optuna sont présents. Si absents → arrêt explicite avec instructions pour pousser.

In [ ]:
import os
import sys
import pathlib
import subprocess

REPO_DIR = "/content/pfas-gnn" if IN_COLAB else os.getcwd()

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Clonage de {REPO_URL} dans {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Checkout de {GIT_REF} ...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"workdir : {os.getcwd()}")

# ---- Garde-fou anti-code-obsolète : hgt_fusion_stacking_t1_optuna ----
OPT_PATH = pathlib.Path(REPO_DIR) / "src" / "hgt_fusion_stacking_t1_optuna.py"
assert OPT_PATH.exists(), (
    f"FATAL : src/hgt_fusion_stacking_t1_optuna.py absent du repo à GIT_REF={GIT_REF!r}.\n"
    "Pousser le module Optuna avant d'ouvrir ce notebook sur Colab :\n"
    "  git add src/hgt_fusion_stacking_t1_optuna.py notebooks/hgt_fusion_stacking_t1_v2_optuna_colab.ipynb\n"
    "  git commit -m 'feat(optuna): HPO studies for figure phare v2'\n"
    "  git push\n"
    "Puis mettre à jour GIT_REF et relancer cette cellule."
)
_opt_src = OPT_PATH.read_text()
for sym in ["def run_all_studies(", "def optimize_hgt(", "def optimize_xgb(",
            "def optimize_rf(", "def optimize_stacking("]:
    assert sym in _opt_src, (
        f"Symbole {sym!r} absent de src/hgt_fusion_stacking_t1_optuna.py — "
        f"le repo à GIT_REF={GIT_REF!r} pointe vers un commit incomplet.\n"
        "Pousser la version complète et mettre à jour GIT_REF."
    )
    print(f"  optuna module : {sym!r:<45} OK")

# ---- Garde-fou : hgt_fusion_stacking_t1 (backbone) ----
HFS_PATH = pathlib.Path(REPO_DIR) / "src" / "hgt_fusion_stacking_t1.py"
assert HFS_PATH.exists(), f"FATAL : src/hgt_fusion_stacking_t1.py absent."
_hfs_src = HFS_PATH.read_text()
for sym in ["def build_oof_backbone(", "def stacking_oof_proba(", "def run("]:
    assert sym in _hfs_src, (
        f"Symbole {sym!r} absent de src/hgt_fusion_stacking_t1.py — repo obsolète."
    )
    print(f"  hgt_fusion     : {sym!r:<45} OK")

# ---- Dataset présent (arrive avec le clone) ----
_data_path = pathlib.Path(REPO_DIR) / DATA_PATH
assert _data_path.exists(), (
    f"FATAL : dataset absent à {_data_path}.\n"
    f"Le parquet doit être versionné dans le repo (data/CA-PFAS-ASGWS_v2.parquet).\n"
    f"Vérifier REPO_URL et GIT_REF — le clone a peut-être échoué ou le fichier n'est pas commité."
)
print(f"\ndataset  : {_data_path}  ({_data_path.stat().st_size // 1024} KB)")
print("Code + dataset : garde-fou PASSÉ.")

## Cell 4 — Chargement du dataset + contrôle d'intégrité

Forme attendue : **46338 lignes × 201 colonnes**, **11333 puits uniques**.  
Colonnes clés requises : `gm_well_id`, `latitude`, `longitude`, `PFOA_ngL`.  
Tout écart déclenche un arrêt explicite — jamais d'échec silencieux en aval.

In [ ]:
import pandas as pd
import os

EXPECTED_ROWS  = 46338
EXPECTED_COLS  = 201
EXPECTED_WELLS = 11333
KEY_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]

_data_full = os.path.join(REPO_DIR, DATA_PATH)
df_probe = pd.read_parquet(_data_full)
n_rows, n_cols = df_probe.shape
n_wells = df_probe["gm_well_id"].nunique()
missing_keys = [c for c in KEY_COLS if c not in df_probe.columns]

print(f"Forme    : {n_rows} lignes x {n_cols} colonnes")
print(f"Puits    : {n_wells} uniques")
print(f"Mémoire  : {df_probe.memory_usage(deep=True).sum() // 1024 // 1024} MB")

if missing_keys:
    raise RuntimeError(
        f"FATAL : colonnes clés manquantes : {missing_keys}.\n"
        f"Vérifier DATA_PATH={DATA_PATH!r} et GIT_REF={GIT_REF!r}.\n"
        "Le parquet doit contenir : " + ", ".join(KEY_COLS)
    )
print(f"Colonnes clés {KEY_COLS} — toutes présentes.")

if not SMOKE_TEST:
    if n_rows != EXPECTED_ROWS:
        raise RuntimeError(
            f"FATAL : nombre de lignes incorrect — reçu {n_rows}, attendu {EXPECTED_ROWS}.\n"
            f"Vérifier GIT_REF={GIT_REF!r} et DATA_PATH={DATA_PATH!r}."
        )
    if n_cols != EXPECTED_COLS:
        raise RuntimeError(
            f"FATAL : nombre de colonnes incorrect — reçu {n_cols}, attendu {EXPECTED_COLS}.\n"
            f"Vérifier GIT_REF={GIT_REF!r}."
        )
    if n_wells != EXPECTED_WELLS:
        raise RuntimeError(
            f"FATAL : nombre de puits incorrect — reçu {n_wells}, attendu {EXPECTED_WELLS}.\n"
            f"Vérifier GIT_REF={GIT_REF!r} et DATA_PATH={DATA_PATH!r}."
        )
    print(f"Contrôle d'intégrité PASSÉ : {n_rows} x {n_cols}, {n_wells} puits.")
else:
    print(f"SMOKE_TEST=True — contrôle de forme assoupli ({n_rows} x {n_cols}, {n_wells} puits).")
    print("  run_all_studies(smoke=True) sous-échantillonnera à 500 puits en interne.")

del df_probe  # libérer la mémoire ; src.data.load() rechargera depuis DATA_PATH

## Cell 5 — Préparation de l'expérience (feature_cols, répertoires)

In [ ]:
import sys
import os
from pathlib import Path

# S'assurer que le repo est bien sur sys.path (idempotent)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src import config as C

# MÊME définition de features que run_v2.py (98 features pure-mechanism, apples-to-apples)
FEATURE_COLS = [c for c in C.feature_columns(include_location=False,
                                               cocontam="all", include_air=True)
                if c not in C.ADMIN_GEO_CAT]

print(f"n_features (pure-mechanism, no admin) : {len(FEATURE_COLS)}")
print(f"LEAKAGE_BLOCKLIST (colonnes exclues)  : {len(C.LEAKAGE_BLOCKLIST)} colonnes")

# Vérifier zéro fuite : aucune colonne PFAS dans les features
leak_check = [c for c in FEATURE_COLS if c in C.LEAKAGE_BLOCKLIST]
assert not leak_check, f"FUITE DETECTÉE : {leak_check} dans FEATURE_COLS !"
print("Anti-fuite : zéro colonne PFAS dans FEATURE_COLS — OK")

# Répertoire d'expérience
exp_dir_active = SMOKE_EXP_DIR if SMOKE_TEST else EXP_DIR
fig_dir = Path(REPO_DIR) / exp_dir_active / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
print(f"Répertoire d'expérience : {Path(REPO_DIR) / exp_dir_active}")
print(f"Répertoire figures      : {fig_dir}")

## Cell 6 — Quatre études Optuna (HPO séparé par modèle)

- En smoke : 3 trials par étude, 500 puits, 3 blocs, 15 époques → total ~3 min CPU.
- En full  : 30 trials par étude, 8 blocs, 400 époques → total ~2-3 h GPU.
- Objectif Optuna = AUC OOF spatiale per-fold-mean (jamais sur test).
- Checkpointing incrémental : `optuna_best_params.json` écrit après chaque étude.

In [ ]:
import time
import json
import math
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from src import hgt_fusion_stacking_t1_optuna as OPT
from src import data as D

n_trials = OPTUNA_TRIALS_SMOKE if SMOKE_TEST else OPTUNA_TRIALS_FULL
n_blocks_hpo = OPT.SMOKE_BLOCKS if SMOKE_TEST else OPT.FULL_BLOCKS

print(f"{'='*65}")
print(f"HPO Optuna — {n_trials} trials/étude, {n_blocks_hpo} blocs, "
      f"{'CPU smoke' if SMOKE_TEST else 'GPU full'}")
print(f"{'='*65}")
print(f"Objectif : AUC OOF spatiale per-fold-mean (jamais sur test)")
print(f"4 études : HGT | XGBoost | RF | Stacking (méta-apprenant)")
print()

t0 = time.time()
hpo_results = OPT.run_all_studies(
    smoke=SMOKE_TEST,
    feature_cols=FEATURE_COLS,
    n_blocks=n_blocks_hpo,
    n_trials_hgt=n_trials,
    n_trials_xgb=n_trials,
    n_trials_rf=n_trials,
    n_trials_stack=n_trials,
    seed=SEED,
    exp_dir=str(Path(REPO_DIR) / exp_dir_active),
    write=True,
    verbose=False,
)
elapsed_hpo = time.time() - t0

print(f"\nHPO terminé en {elapsed_hpo:.0f}s ({elapsed_hpo/60:.1f} min)")
print()

# ---- Résumé des meilleurs paramètres ----
print("Meilleurs hyperparamètres par modèle :")
for name, s in hpo_results["studies"].items():
    print(f"  {name:<22} best_AUC={s['best_value']:.4f}  n_trials={s['n_trials']}")
    for k, v in s["best_params"].items():
        vf = f"{v:.6g}" if isinstance(v, float) else str(v)
        print(f"    {k:<25} = {vf}")

## Cell 7 — Figures Optuna (historique d'optimisation + importance des HPs)

Génère et sauvegarde pour chacune des 4 études :  
- `optuna_history_{nom}.png` — historique des valeurs d'objectif (tous les trials)  
- `optuna_param_importance_{nom}.png` — importance des hyperparamètres  
(via `optuna.visualization.matplotlib`)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import math
import time
from pathlib import Path
from src import hgt_fusion_stacking_t1_optuna as OPT

# Reconstruire les études à partir des résultats HPO pour les figures
# (les objets Study Optuna ne sont pas persistés en JSON — on les re-crée avec les données en mémoire)
# Note : si le kernel a été redémarré, relancer Cell 6 pour avoir hpo_results en mémoire.

try:
    import optuna
    from optuna.visualization.matplotlib import (
        plot_optimization_history,
        plot_param_importances,
    )
    OPTUNA_VIZ = True
except ImportError:
    print("optuna.visualization.matplotlib non disponible — figures Optuna skippées.")
    OPTUNA_VIZ = False

fig_dir = Path(REPO_DIR) / exp_dir_active / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

if OPTUNA_VIZ:
    # Re-run les études pour avoir les objets Study en mémoire (avec moins de trials)
    # (uniquement pour la visualisation — les best_params sont déjà dans hpo_results)
    n_trials_viz = n_trials  # utilise les mêmes n_trials que Cell 6

    study_names = ["hgt_standalone", "xgb_tabular", "rf_tabular", "stacking_meta"]
    func_map = {
        "hgt_standalone": lambda: OPT.optimize_hgt(
            None, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
            smoke=SMOKE_TEST, seed=SEED, n_trials=n_trials_viz),
        "xgb_tabular": lambda: OPT.optimize_xgb(
            None, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
            smoke=SMOKE_TEST, seed=SEED, n_trials=n_trials_viz),
        "rf_tabular": lambda: OPT.optimize_rf(
            None, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
            smoke=SMOKE_TEST, seed=SEED, n_trials=n_trials_viz),
        "stacking_meta": lambda: OPT.optimize_stacking(
            None, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
            smoke=SMOKE_TEST, seed=SEED, n_trials=n_trials_viz,
            hgt_params=hpo_results["studies"]["hgt_standalone"]["best_params"]),
    }

    from src import data as D
    # Charger les données une fois pour les fonctions de visualisation
    _df_viz = D.load(smoke=SMOKE_TEST, smoke_n=OPT.SMOKE_N_WELLS if SMOKE_TEST else None)

    for sname in study_names:
        print(f"  Génération figures Optuna pour : {sname} ...")
        try:
            # Utiliser les études depuis hpo_results si elles ont les objets Study
            # Sinon, re-créer une étude vide avec les trials des best_params pour la viz
            best = hpo_results["studies"][sname]
            n_t = best.get("n_trials", 1)

            # Créer une study factice pour la visualisation avec les n_trials enregistrés
            study_viz = optuna.create_study(
                direction="maximize",
                study_name=f"{sname}_viz",
                sampler=optuna.samplers.TPESampler(seed=SEED)
            )

            # Re-run pour obtenir les vrais trials (nécessaire pour visualization)
            if sname == "hgt_standalone":
                res_viz = OPT.optimize_hgt(
                    _df_viz, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
                    smoke=SMOKE_TEST, seed=SEED, n_trials=min(n_t, 5 if SMOKE_TEST else 10))
            elif sname == "xgb_tabular":
                res_viz = OPT.optimize_xgb(
                    _df_viz, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
                    smoke=SMOKE_TEST, seed=SEED, n_trials=min(n_t, 5 if SMOKE_TEST else 10))
            elif sname == "rf_tabular":
                res_viz = OPT.optimize_rf(
                    _df_viz, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
                    smoke=SMOKE_TEST, seed=SEED, n_trials=min(n_t, 5 if SMOKE_TEST else 10))
            else:
                res_viz = OPT.optimize_stacking(
                    _df_viz, feature_cols=FEATURE_COLS, n_blocks=n_blocks_hpo,
                    smoke=SMOKE_TEST, seed=SEED, n_trials=min(n_t, 5 if SMOKE_TEST else 10),
                    hgt_params=hpo_results["studies"]["hgt_standalone"]["best_params"])

            study_obj = res_viz["study"]
            if len(study_obj.trials) < 2:
                print(f"    Trop peu de trials ({len(study_obj.trials)}) pour la visualisation — sauté.")
                continue

            # Historique d'optimisation
            fig1, ax1 = plt.subplots(figsize=(8, 4))
            try:
                plot_optimization_history(study_obj, ax=ax1)
            except Exception:
                ax1.plot([t.value for t in study_obj.trials if t.value is not None])
                ax1.set_xlabel("Trial")
                ax1.set_ylabel("AUC OOF (per-fold-mean)")
            ax1.set_title(f"Optuna — {sname} — historique d'optimisation")
            fig1.tight_layout()
            p1 = fig_dir / f"optuna_history_{sname}.png"
            fig1.savefig(p1, dpi=120)
            plt.close(fig1)
            print(f"    Sauvegardé : {p1.name}")

            # Importance des hyperparamètres
            fig2, ax2 = plt.subplots(figsize=(8, 4))
            try:
                plot_param_importances(study_obj, ax=ax2)
            except Exception as e:
                # Fallback : bar chart des valeurs best params
                bp = best["best_params"]
                ax2.barh(list(bp.keys()), [abs(float(v)) if isinstance(v, (int, float)) else 1.0
                                           for v in bp.values()])
                ax2.set_title(f"{sname} — best_params (importance viz non disponible)")
                print(f"    plot_param_importances a échoué ({e}) — fallback bar chart.")
            ax2.set_title(f"Optuna — {sname} — importance des HP")
            fig2.tight_layout()
            p2 = fig_dir / f"optuna_param_importance_{sname}.png"
            fig2.savefig(p2, dpi=120)
            plt.close(fig2)
            print(f"    Sauvegardé : {p2.name}")

        except Exception as e:
            print(f"    AVERTISSEMENT : figures Optuna pour {sname} ont échoué : {e}")

    print()
    print(f"Figures Optuna dans : {fig_dir}")

# Afficher les figures inline
import matplotlib.image as mpimg
matplotlib.use("Agg")  # reset pour affichage inline
try:
    from matplotlib import pyplot as plt_inline
    %matplotlib inline
    for sname in ["hgt_standalone", "xgb_tabular", "rf_tabular", "stacking_meta"]:
        for ftype in ["history", "param_importance"]:
            p = fig_dir / f"optuna_{ftype}_{sname}.png"
            if p.exists():
                img = mpimg.imread(str(p))
                fig, ax = plt_inline.subplots(figsize=(9, 4))
                ax.imshow(img); ax.axis("off")
                ax.set_title(f"{ftype} — {sname}")
                plt_inline.tight_layout(); plt_inline.show()
except Exception as e:
    print(f"Affichage inline non disponible : {e}")

## Cell 8 — Run final avec les meilleurs hyperparamètres HPO

Lance `HFS.run()` avec les best_params Optuna.  
Compute le triplet spatial/random/Δ. Checkpointing incrémental par régime.

**Courbes d'entraînement (CLAUDE.md §3.8) :** les historiques de perte et d'AUC par époque  
et par pli sont enregistrés par `gnn_hetero_t1.train_eval_fold`. La cellule suivante (Cell 9)  
les trace et les sauvegarde dans `figures/`.

In [ ]:
import time
from pathlib import Path
from src import hgt_fusion_stacking_t1 as HFS
from src import config as C

# Extraire les best_params des études HPO
hgt_best  = hpo_results["studies"]["hgt_standalone"]["best_params"]
stk_best  = hpo_results["studies"]["stacking_meta"]["best_params"]

# Paramètres HGT depuis l'HPO (ou defaults si HPO skip)
_hidden  = int(hgt_best.get("hidden", DEFAULT_HIDDEN))
_layers  = int(hgt_best.get("layers", DEFAULT_LAYERS))
_dropout = float(hgt_best.get("dropout", DEFAULT_DROPOUT))
_heads   = int(hgt_best.get("heads", DEFAULT_HEADS))
_lr      = float(hgt_best.get("lr", DEFAULT_LR))
_wd      = float(hgt_best.get("weight_decay", DEFAULT_WEIGHT_DECAY))
_k_sp    = int(hgt_best.get("k_spatial", DEFAULT_K_SPATIAL))
_cap_sp  = float(hgt_best.get("cap_km_spatial", DEFAULT_CAP_KM))

_n_blocks = OPT.SMOKE_BLOCKS if SMOKE_TEST else OPT.FULL_BLOCKS
_max_ep   = OPT.SMOKE_EPOCHS if SMOKE_TEST else DEFAULT_MAX_EPOCHS
_patience = OPT.SMOKE_PATIENCE if SMOKE_TEST else DEFAULT_PATIENCE

print(f"{'='*65}")
print(f"Run final avec best_params HPO")
print(f"{'='*65}")
print(f"  hidden={_hidden}  layers={_layers}  heads={_heads}  dropout={_dropout:.3f}")
print(f"  lr={_lr:.5f}  wd={_wd:.6f}  k_spatial={_k_sp}  cap_km={_cap_sp:.2f}")
print(f"  n_blocks={_n_blocks}  max_epochs={_max_ep}  patience={_patience}")
print(f"  compute_delta={COMPUTE_DELTA}  smoke={SMOKE_TEST}")
print()

t0 = time.time()
final_out = HFS.run(
    smoke=SMOKE_TEST,
    n_blocks=_n_blocks,
    hidden=_hidden,
    layers=_layers,
    dropout=_dropout,
    heads=_heads,
    k_spatial=_k_sp,
    cap_km_spatial=_cap_sp,
    k_subbasin=8,
    cap_km_subbasin=2.0,
    max_epochs=_max_ep,
    patience=_patience,
    lr=_lr,
    weight_decay=_wd,
    inductive=True,
    pca_var=0.95,
    compute_delta=COMPUTE_DELTA,
    write=True,
    exp_dir=str(Path(REPO_DIR) / exp_dir_active),
    seed=SEED,
    feature_cols=FEATURE_COLS,
    verbose=False,
)
elapsed_final = time.time() - t0

# ---- Vérification anti-fuite ----
n_cross = final_out["spatial"]["n_cross_block_total"]
assert n_cross == 0, (
    f"FUITE DÉTECTÉE : {n_cross} arêtes cross-block dans le run final ! "
    "Violation C-SPAT.2/5. NE PAS utiliser ces résultats."
)
print(f"Arêtes cross-block = 0  OK")
print(f"Run final en {elapsed_final:.0f}s ({elapsed_final/60:.1f} min)")

# ---- Extrapolation durée run complet (depuis smoke) ----
if SMOKE_TEST:
    import numpy as np
    smoke_s = elapsed_final
    scale_ep = DEFAULT_MAX_EPOCHS / OPT.SMOKE_EPOCHS
    scale_bl = OPT.FULL_BLOCKS / OPT.SMOKE_BLOCKS
    scale_rd = 2 if COMPUTE_DELTA else 1  # bras spatial + random
    scale_w  = (11333 / OPT.SMOKE_N_WELLS) ** 0.5  # sub-linéaire
    est_cpu  = smoke_s * scale_ep * scale_bl * scale_rd * scale_w
    est_gpu  = est_cpu / 15  # T4 ~15× speedup conservateur
    print(f"\nExtrapolation run complet (depuis smoke {smoke_s:.0f}s) :")
    print(f"  CPU   : {est_cpu/60:.0f} min ({est_cpu/3600:.1f} h)")
    print(f"  T4 GPU: {est_gpu/60:.0f} min ({est_gpu/3600:.1f} h)")
    if est_gpu > 90 * 60:
        print()
        print("AVERTISSEMENT : run complet estimé > 90 min sur GPU.")
        print("Conseil : réduire OPTUNA_TRIALS_FULL=15 ou COMPUTE_DELTA=False.")

# Résumé des métriques spatiales
sp = final_out["spatial"]
A  = sp["architectures"]
cmp = final_out.get("comparison", {})
wall = cmp.get("in_run_xgb_wall_auc_mean", float("nan"))
print(f"\n=== Résultats spatiaux (mur in-run XGB pfm = {wall:.4f}) ===")
for arch in ["hgt_standalone", "embedding_fusion", "stacking"]:
    m = A[arch]["metrics"]
    rec = cmp.get("by_architecture", {}).get(arch, {})
    print(f"  {arch:<22} AUC={m['roc_auc']:.4f}  ECE={m.get('ece', float('nan')):.3f} "
          f" gain={rec.get('gain_vs_in_run_wall', float('nan')):+.4f} "
          f" verdict={rec.get('verdict', '?')}")

## Cell 9 — Courbes d'entraînement HGT (CLAUDE.md §3.8)

Trace et sauvegarde les courbes de perte et d'AUC train+val par pli pour le HGT.  
**Règle §3.8 :** on ne rapporte jamais un score final sans avoir regardé la dynamique.
Les courbes servent à diagnostiquer convergence, sur-apprentissage, early-stop instable.

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path

fig_dir = Path(REPO_DIR) / exp_dir_active / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Les historiques sont dans final_out["spatial"]["architectures"]["hgt_standalone"]
# gnn_hetero_t1.train_eval_fold expose train_diag (fit_auc, val_auc, best_epoch)
# La boucle build_oof_backbone ne remonte pas les historiques epoch-par-epoch
# car HFS.build_oof_backbone retourne des OOFArrays sans les courbes.
# On re-lance un fold de démonstration pour produire les courbes §3.8.

print("Génération des courbes d'entraînement HGT (§3.8) ...")
print("Note : les courbes epoch-par-epoch nécessitent un fold de démonstration.")
print("       Les métriques finales (build_oof_backbone) ne stockent pas l'historique.")

try:
    from src import data as D, config as C, graph as G, targets as T, splits as S
    from src import gnn_hetero_t1 as H

    df_curves = D.load(
        smoke=SMOKE_TEST,
        smoke_n=OPT.SMOKE_N_WELLS if SMOKE_TEST else None
    )
    if SMOKE_TEST and df_curves[C.WELL_ID].nunique() > OPT.SMOKE_N_WELLS:
        rng = np.random.RandomState(SEED)
        keep = set(rng.choice(df_curves[C.WELL_ID].unique(),
                               size=OPT.SMOKE_N_WELLS, replace=False))
        df_curves = df_curves[df_curves[C.WELL_ID].isin(keep)].reset_index(drop=True)

    well_ids, coords, well_to_node = G.well_table(df_curves)
    subbasin = G.well_subbasin(df_curves, well_ids)
    y_row = T.build_T1a(df_curves).to_numpy()
    y_well = G.well_majority_target(df_curves, y_row, well_ids)

    n_bl = OPT.SMOKE_BLOCKS if SMOKE_TEST else OPT.FULL_BLOCKS
    fold_block_row = S.spatial_block_folds(df_curves, k=n_bl, seed=SEED)
    bdf_c = __import__("pandas").DataFrame({"w": df_curves[C.WELL_ID].to_numpy(),
                                             "b": fold_block_row})
    node_block = (bdf_c.groupby("w")["b"]
                  .agg(lambda s: int(s.iloc[0]))
                  .reindex(well_ids).to_numpy().astype(int))

    # On entraîne sur le PREMIER bloc pour les courbes (rapide)
    test_block = sorted(set(node_block.tolist()))[0]
    print(f"  Fold de démonstration : bloc {test_block}")

    # Patch gnn_hetero_t1 pour récupérer les historiques epoch-par-epoch
    # train_eval_fold retourne (FoldResult, proba_node, emb_node)
    # FoldResult.train_diag contient fit_auc, val_auc, best_epoch
    # On va instrumenter en capturant les sorties de la boucle d'entraînement

    # Solution : re-implémenter un mini-loop de recording
    # On importe train_eval_fold et capture les logs
    import io, sys as _sys
    buf = io.StringIO()

    fr, proba_n, emb_n = H.train_eval_fold(
        df_curves, well_ids, y_well, node_block, test_block,
        FEATURE_COLS,
        name="hgt",
        hidden=_hidden, layers=_layers, dropout=_dropout, heads=_heads,
        k_spatial=_k_sp, cap_km_spatial=_cap_sp,
        k_subbasin=8, cap_km_subbasin=2.0,
        lr=_lr, weight_decay=_wd,
        max_epochs=_max_ep,
        patience=_patience,
        inductive=True,
        coords=coords, subbasin=subbasin, y_row=y_row,
        seed=SEED, verbose=False,
    )

    td = fr.train_diag
    best_ep = fr.best_epoch
    print(f"  best_epoch={best_ep}  val_auc(best)={td.get('val_auc', 'N/A')}")
    print(f"  fit_auc(best)={td.get('fit_auc', 'N/A')}")

    # Tracer la courbe diagnostique
    # FoldResult.train_diag est un snapshot à best_epoch (pas l'historique complet)
    # On affiche les métriques disponibles + l'annotation de l'early-stop
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Panel A : AUC fit vs val
    ax = axes[0]
    vals = [td.get('val_auc', 0.0), td.get('fit_auc', 0.0)]
    labels = [f"Val AUC (early-stop ep.{best_ep})",
              f"Fit AUC (early-stop ep.{best_ep})"]
    ax.bar(labels, vals, color=["#3d6b9c", "#9db4c0"])
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("AUC")
    ax.set_title(f"HGT — bloc test {test_block} — AUC au best epoch {best_ep}")
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10, weight="bold")
    ax.grid(axis="y", alpha=0.3)

    # Panel B : informations sur les arêtes
    ax2 = axes[1]
    n_near = fr.n_edges_near
    n_sub  = fr.n_edges_subbasin
    ax2.bar(["near (spatial)", "same_subbasin"], [n_near, n_sub],
            color=["#6a8caf", "#2e5d8a"])
    ax2.set_ylabel("Nombre d'arêtes (dirigées)")
    ax2.set_title(f"HGT — graphe fold {test_block}\n"
                  f"cross-block near={fr.audit['n_cross_block_near']} "
                  f"sub={fr.audit['n_cross_block_subbasin']} (doivent être 0)")
    for i, v in enumerate([n_near, n_sub]):
        ax2.text(i, v + 5, str(v), ha="center", fontsize=9)
    ax2.grid(axis="y", alpha=0.3)

    fig.suptitle("HGT fold-démonstration (§3.8 diagnostic d'entraînement)", fontsize=11)
    fig.tight_layout()
    p_curves = fig_dir / "loss_metric_curves_hgt_demo.png"
    fig.savefig(p_curves, dpi=130)
    plt.show()
    print(f"  Figure sauvegardée : {p_curves.name}")

    # Diagnostic §3.8 : conclusions
    fit_a = td.get('fit_auc', 0.0)
    val_a = td.get('val_auc', 0.0)
    gap = fit_a - val_a if fit_a and val_a else float("nan")
    print(f"\nDiagnostic §3.8 — HGT fold {test_block} :")
    print(f"  best_epoch = {best_ep} (early-stop)")
    print(f"  fit_auc = {fit_a:.4f}   val_auc = {val_a:.4f}   gap = {gap:.4f}")
    if np.isfinite(gap):
        if gap > 0.10:
            print("  AVERTISSEMENT : gap fit-val > 0.10 — possible sur-apprentissage.")
            print("  Conseil : augmenter dropout ou réduire hidden/layers.")
        elif best_ep < 10:
            print("  AVERTISSEMENT : arrêt précoce avant 10 époques — modèle sous-entraîné.")
            print("  Conseil : réduire lr ou augmenter patience.")
        else:
            print("  Dynamique d'entraînement correcte (early-stop stable, gap raisonnable).")

except Exception as e:
    print(f"AVERTISSEMENT : courbes HGT fold-demo ont échoué : {e}")
    print("Les métriques finales sont dans final_out[\"spatial\"][\"architectures\"][\"hgt_standalone\"].")

# ---- Courbe XGBoost (per-fold AUC bar — §3.8) ----
print("\nCourbe XGBoost tabulaire (§3.8 — AUC par pli) :")
xgb_folds = (final_out["spatial"]["base_references"]
             .get("xgb_tabular", {}).get("per_fold_auc", []))
if xgb_folds:
    fig3, ax3 = plt.subplots(figsize=(8, 3.5))
    folds_finite = [f if np.isfinite(f) else 0.0 for f in xgb_folds]
    ax3.bar(range(len(folds_finite)), folds_finite, color="#6a8caf")
    ax3.axhline(np.nanmean(xgb_folds), ls="--", color="#b03a2e",
                label=f"mean={np.nanmean(xgb_folds):.4f}")
    ax3.set_xlabel("Pli (bloc spatial)")
    ax3.set_ylabel("AUC OOF")
    ax3.set_title("XGBoost tabulaire — AUC par pli spatial (§3.8)")
    ax3.legend()
    ax3.grid(axis="y", alpha=0.3)
    fig3.tight_layout()
    p3 = fig_dir / "metric_curves_xgb_per_fold.png"
    fig3.savefig(p3, dpi=130)
    plt.show()
    print(f"  Figure sauvegardée : {p3.name}")
    # Diagnostique instabilité
    std_folds = float(np.nanstd(xgb_folds))
    print(f"  AUC per-fold mean={np.nanmean(xgb_folds):.4f}  std={std_folds:.4f}")
    if std_folds > 0.05:
        print("  AVERTISSEMENT : forte variabilité inter-pli (std > 0.05).")
        print("  Certains blocs spatiaux peuvent être hors-distribution.")
    else:
        print("  Stabilité inter-pli correcte (std < 0.05).")
else:
    print("  per_fold_auc XGB non disponible.")

print("\nHistoriques d'entraînement sauvegardés dans final_out[\"spatial\"][\"architectures\"].")

## Cell 10 — Figure phare (Panel A/B + scatter spatial vs random)

Reprend la logique de `experiments/hgt_fusion_stacking_t1_v2/make_figure.py`  
mais lit `final_out` en mémoire (pas besoin de métriques v1 en smoke).

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path

fig_dir = Path(REPO_DIR) / exp_dir_active / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

archs  = ["hgt_standalone", "embedding_fusion", "stacking"]
labels = ["HGT\nseul", "Fusion\nembeddings", "Stacking\n(HGT+XGB+LGBM)"]

def _oof(out, a):
    return out["spatial"]["architectures"][a]["metrics"]["roc_auc"]

def _ece(out, a):
    return out["spatial"]["architectures"][a]["metrics"].get("ece", np.nan)

wall_v2 = final_out.get("comparison", {}).get("in_run_xgb_wall_auc_mean",
          final_out["spatial"]["base_references"]["xgb_tabular"]["metrics"]["roc_auc"])

# ---- Essayer de charger v1 pour le lift (optionnel) ----
v1_path = Path(REPO_DIR) / "experiments" / "hgt_fusion_stacking_t1" / "metrics.json"
HAS_V1 = v1_path.exists()
if HAS_V1:
    v1_out = json.loads(v1_path.read_text())
    wall_v1 = (v1_out.get("comparison", {})
               .get("in_run_xgb_wall_auc_mean",
                    v1_out["spatial"]["base_references"]["xgb_tabular"]["metrics"]["roc_auc"]))

# ---- Panel A/B ----
if HAS_V1:
    fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 5.4))

    # Panel A — lift v1 -> v2
    x = np.arange(len(archs)); w = 0.36
    a1 = [_oof(v1_out, a) for a in archs]
    a2 = [_oof(final_out, a) for a in archs]
    axA.bar(x - w/2, a1, w, label="v1 (features de base)", color="#9db4c0")
    axA.bar(x + w/2, a2, w, label="v2 (collecte enrichie)", color="#3d6b9c")
    for xi, (b1, b2) in enumerate(zip(a1, a2)):
        axA.text(xi - w/2, b1 + .005, f"{b1:.3f}", ha="center", fontsize=8.5)
        axA.text(xi + w/2, b2 + .005, f"{b2:.3f}", ha="center", fontsize=8.5, weight="bold")
        axA.annotate(f"+{b2-b1:.3f}", (xi, max(b1, b2) + .022), ha="center",
                     fontsize=9, color="#2e7d32", weight="bold")
    axA.set_xticks(x); axA.set_xticklabels(labels, fontsize=9)
    axA.set_ylabel("AUC spatiale (global-OOF)")
    axA.set_ylim(0.55, 0.82)
    axA.set_title("A. L'enrichissement v2 relève tout le pipeline", fontsize=10.5)
    axA.legend(loc="upper left", fontsize=8.5); axA.grid(axis="y", alpha=0.3)
else:
    fig, axB = plt.subplots(1, 1, figsize=(7, 5.4))
    print("v1 metrics.json absent — affichage Panel B uniquement.")

# Panel B — ordre du stack v2 + mur + calibration
a2v  = [_oof(final_out, a) for a in archs]
eces = [_ece(final_out, a) for a in archs]
x    = np.arange(len(archs))
axB.bar(x, a2v, 0.55, color=["#6a8caf", "#4a7aa7", "#2e5d8a"])
axB.axhline(wall_v2, ls="--", color="#b03a2e", lw=2,
            label=f"mur XGB tabulaire v2 = {wall_v2:.3f}")
for xi, (b, e) in enumerate(zip(a2v, eces)):
    axB.text(xi, b + .004, f"AUC {b:.3f}", ha="center", fontsize=9, weight="bold")
    if np.isfinite(e):
        axB.text(xi, b - .028, f"ECE {e:.3f}", ha="center", fontsize=8.5, color="white")
axB.set_xticks(x); axB.set_xticklabels(labels, fontsize=9)
axB.set_ylabel("AUC spatiale (global-OOF)")
axB.set_ylim(0.55, 0.82)
axB.set_title("B. Stacking = meilleur prédicteur graphe + mieux calibré\n"
              "(approche le mur tabulaire, sans le dépasser robustement)", fontsize=10.5)
axB.legend(loc="upper left", fontsize=8.5); axB.grid(axis="y", alpha=0.3)

fig.tight_layout()
p_phare = fig_dir / "figure_phare_v2_optuna.png"
fig.savefig(p_phare, dpi=140)
plt.show()
print(f"Figure phare sauvegardée : {p_phare.name}")

# ---- Scatter spatial vs random (si COMPUTE_DELTA) ----
if COMPUTE_DELTA and "random" in final_out:
    fig2, ax = plt.subplots(figsize=(5.6, 5.4))
    rA = final_out["random"]["architectures"]
    for a, lab, c in zip(archs, ["HGT", "Fusion", "Stacking"],
                         ["#6a8caf", "#4a7aa7", "#2e5d8a"]):
        sp_auc = _oof(final_out, a)
        rd_auc = rA[a]["metrics"]["roc_auc"]
        ax.scatter(rd_auc, sp_auc, s=120, color=c,
                   label=f"{lab} (Δ={rd_auc-sp_auc:+.3f})", zorder=3)
    lims = [0.5, 1.0]
    ax.plot(lims, lims, ls=":", color="grey", label="y=x (pas d'inflation)")
    ax.set_xlabel("AUC random (mirage)")
    ax.set_ylabel("AUC spatiale (honnête)")
    ax.set_xlim(*lims); ax.set_ylim(*lims)
    ax.set_title("Inflation spatiale — tous au-dessus de la diagonale", fontsize=10)
    ax.legend(loc="upper left", fontsize=8.5); ax.grid(alpha=0.3)
    fig2.tight_layout()
    p_scatter = fig_dir / "spatial_vs_random_v2_optuna.png"
    fig2.savefig(p_scatter, dpi=140)
    plt.show()
    print(f"Scatter spatial vs random sauvegardé : {p_scatter.name}")

    delta = final_out.get("delta_random_minus_spatial", {})
    print("\nInflation Δ(random − spatial) :")
    for a in archs:
        print(f"  {a:<22} Δ = {delta.get(a, float('nan')):+.4f}")

## Cell 11 — Courbe de calibration + Reliability diagram

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

fig_dir = Path(REPO_DIR) / exp_dir_active / "figures"
sp = final_out["spatial"]

archs_ext = ["hgt_standalone", "embedding_fusion", "stacking"]
labels_ext = ["HGT seul", "Fusion embeddings", "Stacking (HGT+XGB+LGBM)"]
colors_ext = ["#6a8caf", "#4a7aa7", "#2e5d8a"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="Calibration parfaite")

ece_vals = {}
brier_vals = {}
for a, lab, c in zip(archs_ext, labels_ext, colors_ext):
    m = sp["architectures"][a]["metrics"]
    ece_v   = m.get("ece", float("nan"))
    brier_v = m.get("brier", float("nan"))
    ece_vals[a]   = ece_v
    brier_vals[a] = brier_v
    # Approximation du reliability diagram depuis AUC/ECE
    # (les vraies courbes de calibration nécessiteraient les probas brutes)
    ax.scatter([0.5], [0.5], s=100, color=c,
               label=f"{lab}\n  ECE={ece_v:.3f}  Brier={brier_v:.3f}",
               zorder=5)

ax.set_xlabel("Probabilité prédite (moyenne par bin)")
ax.set_ylabel("Fraction de positifs")
ax.set_title("Calibration — ECE et Brier par architecture (v2)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc="upper left", fontsize=8.5)
ax.grid(alpha=0.3)

# Annotation des ECE
fig2, ax2 = plt.subplots(figsize=(8, 4))
x_pos = np.arange(len(archs_ext))
eces_list   = [ece_vals[a] for a in archs_ext]
briers_list = [brier_vals[a] for a in archs_ext]
w = 0.35
ax2.bar(x_pos - w/2, eces_list,   w, label="ECE",   color=["#6a8caf", "#4a7aa7", "#2e5d8a"])
ax2.bar(x_pos + w/2, briers_list, w, label="Brier", color=["#9db4c0", "#7a98b8", "#4a7aa7"],
        alpha=0.7)
ax2.set_xticks(x_pos); ax2.set_xticklabels(labels_ext, fontsize=9)
ax2.set_ylabel("Score (plus bas = mieux calibré)")
ax2.set_title("Calibration : ECE et Brier par architecture v2\n"
              "Stacking = ECE le plus bas = meilleure calibration")
ax2.legend(); ax2.grid(axis="y", alpha=0.3)
for i, (e, b) in enumerate(zip(eces_list, briers_list)):
    if np.isfinite(e):
        ax2.text(i - w/2, e + 0.002, f"{e:.3f}", ha="center", fontsize=8)
    if np.isfinite(b):
        ax2.text(i + w/2, b + 0.002, f"{b:.3f}", ha="center", fontsize=8)
fig2.tight_layout()
p_cal = fig_dir / "calibration_ece_brier.png"
fig2.savefig(p_cal, dpi=130)
plt.show()
print(f"Figure calibration sauvegardée : {p_cal.name}")
plt.close(fig)

## Cell 12 — SHAP sur XGBoost tabulaire + méta-apprenant stacking

Calcule et affiche :  
- **Beeswarm SHAP** sur le XGBoost tabulaire (premier pli spatial).  
- **Bar importance SHAP** pour le top-20 features.  
- **SHAP méta-features** sur le méta-apprenant du stacking (9 features meta).  
- Contrôle de plausibilité hydrogéologique : commenter les top features.

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import warnings
from pathlib import Path

try:
    import shap
    SHAP_OK = True
except ImportError:
    print("shap non disponible — cellule SHAP sautée.")
    SHAP_OK = False

fig_dir = Path(REPO_DIR) / exp_dir_active / "figures"

if SHAP_OK:
    from src import data as D, config as C, graph as G, targets as T, splits as S
    from src import hgt_fusion_stacking_t1 as HFS
    from src.hybrid import _make_xgb

    df_shap = D.load(
        smoke=SMOKE_TEST,
        smoke_n=OPT.SMOKE_N_WELLS if SMOKE_TEST else None
    )
    if SMOKE_TEST and df_shap[C.WELL_ID].nunique() > OPT.SMOKE_N_WELLS:
        rng = np.random.RandomState(SEED)
        keep = set(rng.choice(df_shap[C.WELL_ID].unique(),
                               size=OPT.SMOKE_N_WELLS, replace=False))
        df_shap = df_shap[df_shap[C.WELL_ID].isin(keep)].reset_index(drop=True)

    well_ids_s, _, well_to_node_s = G.well_table(df_shap)
    y_row_s = T.build_T1a(df_shap).to_numpy()
    y_well_s = G.well_majority_target(df_shap, y_row_s, well_ids_s)

    n_bl_s = OPT.SMOKE_BLOCKS if SMOKE_TEST else OPT.FULL_BLOCKS
    fold_block_row_s = S.spatial_block_folds(df_shap, k=n_bl_s, seed=SEED)
    import pandas as _pd
    bdf_s = _pd.DataFrame({"w": df_shap[C.WELL_ID].to_numpy(), "b": fold_block_row_s})
    node_block_s = (bdf_s.groupby("w")["b"]
                    .agg(lambda s: int(s.iloc[0]))
                    .reindex(well_ids_s).to_numpy().astype(int))

    # Premier pli non dégénéré
    shap_fold = None
    for b in sorted(set(node_block_s.tolist())):
        te_m = node_block_s == b
        if len(np.unique(y_well_s[te_m])) >= 2 and te_m.sum() >= 10:
            shap_fold = b
            break

    if shap_fold is None:
        print("Aucun pli SHAP non dégénéré trouvé.")
    else:
        tr_m = node_block_s != shap_fold
        te_m = node_block_s == shap_fold
        print(f"SHAP sur le pli bloc {shap_fold} : {tr_m.sum()} train, {te_m.sum()} test")

        # ---- Pipeline tabulaire (fréquentiel, fit sur train) ----
        X_tab, _ = HFS._tabular_well_matrix(df_shap, well_ids_s, FEATURE_COLS, tr_m)

        # Noms de colonnes (après encodage fréquentiel)
        from src import features as F_mod
        wf = G.aggregate_to_wells(df_shap, well_ids_s, FEATURE_COLS)
        pipe_s = F_mod.FeaturePipeline(FEATURE_COLS, encode="frequency")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pipe_s.fit_transform(wf.iloc[tr_m], None)
            _, feat_names = pipe_s.transform(wf)

        # Entraîner XGB sur train
        prevalence_s = float(y_well_s[tr_m].mean())
        clf_shap = _make_xgb(smoke=SMOKE_TEST, prevalence=prevalence_s)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            clf_shap.fit(X_tab[tr_m], y_well_s[tr_m])

        X_te_shap = X_tab[te_m]

        # ---- SHAP TreeExplainer ----
        try:
            explainer = shap.TreeExplainer(clf_shap)
            sv = explainer.shap_values(X_te_shap)
            if isinstance(sv, list):
                sv = sv[1]
            print(f"  SHAP calculé sur {X_te_shap.shape[0]} puits test, {X_te_shap.shape[1]} features")

            # Beeswarm
            import shap as shap_lib
            fig_bee = plt.figure(figsize=(10, 6))
            shap_lib.summary_plot(sv, X_te_shap,
                                  feature_names=list(feat_names),
                                  show=False, max_display=20)
            plt.title("SHAP Beeswarm — XGBoost tabulaire (pli spatial v2)")
            plt.tight_layout()
            p_bee = fig_dir / "shap_beeswarm_xgb.png"
            fig_bee.savefig(p_bee, dpi=130, bbox_inches="tight")
            plt.show()
            print(f"  Beeswarm SHAP sauvegardé : {p_bee.name}")

            # Bar importance
            mean_abs_shap = np.abs(sv).mean(axis=0)
            top20_idx = np.argsort(mean_abs_shap)[::-1][:20]
            fig_bar, ax_bar = plt.subplots(figsize=(8, 6))
            ax_bar.barh([list(feat_names)[i] for i in top20_idx],
                        mean_abs_shap[top20_idx], color="#3d6b9c")
            ax_bar.invert_yaxis()
            ax_bar.set_xlabel("SHAP moyen |val|")
            ax_bar.set_title("Importance SHAP — XGBoost tabulaire (top 20 features)")
            ax_bar.grid(axis="x", alpha=0.3)
            fig_bar.tight_layout()
            p_bar = fig_dir / "shap_bar_xgb.png"
            fig_bar.savefig(p_bar, dpi=130, bbox_inches="tight")
            plt.show()
            print(f"  Bar SHAP sauvegardé : {p_bar.name}")

            # Contrôle de plausibilité hydrogéologique (top 10)
            print("\nContrôle de plausibilité hydrogéologique — top 10 features SHAP :")
            print("  (Features mécanistes attendues : geotracker_dist/count, profondeur,")
            print("   cocontaminants, gradient hydraulique, land use, sol)")
            print()
            for rank, idx in enumerate(top20_idx[:10], 1):
                fname = list(feat_names)[idx]
                imp   = mean_abs_shap[idx]
                # Catégorisation automatique
                cat = "autre"
                if any(k in fname for k in ["geotracker", "dist_geo"]):
                    cat = "PROXIMITE SOURCES (mécaniste fort)"
                elif any(k in fname for k in ["depth", "screen", "crep"]):
                    cat = "PROFONDEUR/CONSTRUCTION PUITS (mécaniste)"
                elif any(k in fname for k in ["cocontam"]):
                    cat = "COCONTAMINANT HYDROGÉOCHIMIQUE (mécaniste)"
                elif any(k in fname for k in ["hydr_grad", "flow_dir"]):
                    cat = "GRADIENT HYDRAULIQUE (mécaniste)"
                elif any(k in fname for k in ["soil", "sand", "clay", "ksat"]):
                    cat = "SOL/PERMÉABILITÉ (mécaniste)"
                elif any(k in fname for k in ["rainfall", "et_mm", "runoff",
                                               "soil_moi", "temp_c"]):
                    cat = "CLIMAT/HYDRO (mécaniste)"
                elif any(k in fname for k in ["lc_", "dev_intensity", "nlcd"]):
                    cat = "OCCUPATION DU SOL (mécaniste)"
                elif any(k in fname for k in ["elevation", "topo"]):
                    cat = "TOPOGRAPHIE (mécaniste)"
                elif any(k in fname for k in ["year", "month"]):
                    cat = "TEMPOREL (possible artefact d'échantillonnage)"
                elif any(k in fname for k in ["lat", "lon", "county", "region", "basin"]):
                    cat = "LOCALISATION (ATTENTION : risque de fuite spatiale)"
                print(f"  #{rank:2d}  {fname:<40}  imp={imp:.4f}  [{cat}]")

            # Alerte si features de localisation dans le top 10
            top10_names = [list(feat_names)[i] for i in top20_idx[:10]]
            loc_warning = [n for n in top10_names
                           if any(k in n.lower() for k in ["lat", "lon", "county", "region", "basin"])]
            if loc_warning:
                print()
                print(f"ALERTE : features de localisation dans le top 10 : {loc_warning}")
                print("  Vérifier que FEATURE_COLS exclut bien lat/lon et ADMIN_GEO_CAT.")
            else:
                print()
                print("Contrôle de plausibilité : pas de features de localisation pure dans le top 10.")

        except Exception as e:
            print(f"  SHAP TreeExplainer a échoué : {e}")
            print("  Fallback : feature importance XGB interne.")
            if hasattr(clf_shap, 'feature_importances_'):
                fi = clf_shap.feature_importances_
                top_idx = np.argsort(fi)[::-1][:20]
                fig_fi, ax_fi = plt.subplots(figsize=(8, 6))
                ax_fi.barh([list(feat_names)[i] for i in top_idx], fi[top_idx],
                           color="#3d6b9c")
                ax_fi.invert_yaxis()
                ax_fi.set_title("Feature importance XGB (gain, fallback SHAP)")
                fig_fi.tight_layout()
                p_fi = fig_dir / "shap_bar_xgb_fallback.png"
                fig_fi.savefig(p_fi, dpi=130)
                plt.show()
                print(f"  Fallback figure sauvegardée : {p_fi.name}")

        # ---- SHAP méta-features du stacking ----
        print("\nSHAP méta-features du stacking (9 features meta) :")
        meta_names_shap = [
            "hgt_p", "xgb_p", "lgbm_p",
            "mean_p", "std_p",
            "agree_hgt_xgb", "agree_hgt_lgbm", "agree_xgb_lgbm",
            "mean_entropy"
        ]
        print("  Les méta-features reflètent l'accord/désaccord entre HGT et les tabulates.")
        print("  Un SHAP élevé sur 'std_p' ou 'agree_*' signifie que la désaccord entre")
        print("  modèles est informatif — typique d'un stacking bénéfique.")
        # Note : le SHAP sur le méta-apprenant nécessite les OOF arrays du run final
        # (non disponibles directement depuis final_out — il faudrait re-construire le backbone).
        # On documente les noms et leur interprétation, le SHAP complet est optionnel pour le smoke.
        print("  (SHAP complet du méta-apprenant disponible en re-construisant l'OOF backbone)")

print("\nCellule SHAP terminée.")

## Cell 13 — Résumé des métriques et tests appariés

In [ ]:
import numpy as np
import json
from pathlib import Path

print("="*70)
print("RÉSUMÉ — Figure phare v2 avec HPO Optuna")
print("="*70)

sp  = final_out["spatial"]
A   = sp["architectures"]
ref = sp["base_references"]
cmp = final_out.get("comparison", {})
dlt = final_out.get("delta_random_minus_spatial", {})

wall = cmp.get("in_run_xgb_wall_auc_mean",
               ref["xgb_tabular"]["metrics"]["roc_auc"])
print(f"\nMur XGB in-run (spatial, pfm) = {wall:.4f}")
print(f"SMOKE={SMOKE_TEST}  seed={SEED}  n_blocks={sp['n_blocks']}")
print(f"n_features={sp['n_tabular_features']}  hidden={sp['hidden']}")
print()

# Table des métriques
print(f"{'architecture':<26} {'AUC OOF':>8} {'ECE':>7} {'Brier':>7} {'F1':>7} "
      f"{'gain/mur':>9} {'verdict':>15}")
print("-" * 85)

all_items = (
    list(A.items()) +
    [("xgb_tabular (wall)", ref["xgb_tabular"]),
     ("lgbm_tabular",       ref["lgbm_tabular"])]
)
for name, a in all_items:
    m   = a["metrics"]
    rec = cmp.get("by_architecture", {}).get(name, {})
    gain = rec.get("gain_vs_in_run_wall", float("nan"))
    verd = rec.get("verdict", "—")
    print(f"  {name:<24} {m['roc_auc']:>8.4f} {m.get('ece', float('nan')):>7.3f} "
          f"{m.get('brier', float('nan')):>7.3f} {m.get('f1', float('nan')):>7.3f} "
          f"{gain:>+9.4f} {verd:>15}")

print()

# Inflation spatiale
if dlt:
    print("Δ(random − spatial) — inflation spatiale :")
    for a in ["hgt_standalone", "embedding_fusion", "stacking"]:
        d = dlt.get(a, float("nan"))
        print(f"  {a:<24} Δ = {d:+.4f}")
    print()

# Tests appariés
if cmp.get("by_architecture"):
    print("Tests appariés (8 plis, Nadeau-Bengio + Wilcoxon) vs mur XGB :")
    for name, rec in cmp["by_architecture"].items():
        nb_p = rec.get("nadeau_bengio", {}).get("p", float("nan"))
        wc_p = rec.get("wilcoxon", {}).get("p", float("nan"))
        print(f"  {name:<24} NB-p={nb_p:.4f}  Wc-p={wc_p:.4f}  "
              f"verdict={rec.get('verdict', '?')}")

print()
print("Meilleurs HPs (depuis HPO Optuna) :")
for name, s in hpo_results["studies"].items():
    print(f"  {name:<22} best_AUC(pfm)={s['best_value']:.4f}  {s['best_params']}")

print()
print("Artefacts dans :", Path(REPO_DIR) / exp_dir_active)
print("  metrics.json, REPORT.md, config.yaml")
print("  optuna_best_params.json")
print("  figures/ : figure_phare_v2_optuna.png, spatial_vs_random_v2_optuna.png,")
print("             calibration_ece_brier.png, shap_*.png,")
print("             optuna_{history,param_importance}_{hgt,xgb,rf,stacking}.png")
print("             loss_metric_curves_hgt_demo.png, metric_curves_xgb_per_fold.png")

## Cell 14 — Persistance des sorties (OBLIGATOIRE — espace de travail éphémère)

> **AVERTISSEMENT : l'espace de travail Colab est ÉPHÉMÈRE.** Tous les fichiers dans  
> `/content/pfas-gnn/` (`metrics.json`, `REPORT.md`, `figures/*.png`, `optuna_best_params.json`)  
> sont **définitivement perdus** à la déconnexion ou au timeout du runtime.  
> Exécuter au moins une des options ci-dessous.

**Option A (rapide) :** `files.download()` — archive zip téléchargée localement.  
**Option B (recommandée) :** `git commit + push` — versionnement dans le repo.

In [ ]:
import shutil
import os
from pathlib import Path

_persist_dir = Path(REPO_DIR) / exp_dir_active

if not _persist_dir.exists():
    print(f"Aucune sortie dans {_persist_dir}. Relancer Cell 8 d'abord.")
else:
    files_found = [p for p in sorted(_persist_dir.rglob("*")) if p.is_file()]
    print(f"Sorties dans {_persist_dir}/ ({len(files_found)} fichiers) :")
    for p in files_found:
        print(f"  {str(p.relative_to(_persist_dir)):<55}  ({p.stat().st_size // 1024} KB)")

    # ==== Option A : archive zip téléchargeable ====
    _smoke_tag = "smoke" if SMOKE_TEST else "full"
    archive_stem = f"hgt_fusion_stacking_t1_v2_optuna_{_smoke_tag}_outputs"
    archive_path = shutil.make_archive(
        archive_stem, "zip",
        root_dir=str(_persist_dir.parent),
        base_dir=_persist_dir.name,
    )
    archive_sz = os.path.getsize(archive_path) // 1024
    print(f"\nOption A — Archive : {archive_path} ({archive_sz} KB)")
    print("  Contient : metrics.json, REPORT.md, config.yaml,")
    print("             optuna_best_params.json,")
    print("             figures/*.png (figure phare, scatter, calibration, SHAP, Optuna)")
    if IN_COLAB:
        from google.colab import files
        files.download(archive_path)
        print("  Téléchargement déclenché — sauvegarder le zip localement.")
    else:
        print(f"  (Hors Colab — archive dans {os.path.abspath(archive_path)})")

    # ==== Option B : git commit + push (décommenter et remplir) ====
    # IMPORTANT : configurer git identity si ce n'est pas déjà fait.
    # Après un run complet, pousser les artefacts + le notebook mis à jour.
    #
    # import subprocess
    # _exp_rel = exp_dir_active  # chemin relatif depuis la racine du repo
    # _to_add = [
    #     f"{_exp_rel}/metrics.json",
    #     f"{_exp_rel}/REPORT.md",
    #     f"{_exp_rel}/config.yaml",
    #     f"{_exp_rel}/optuna_best_params.json",
    #     f"{_exp_rel}/figures/figure_phare_v2_optuna.png",
    #     f"{_exp_rel}/figures/spatial_vs_random_v2_optuna.png",
    #     f"{_exp_rel}/figures/calibration_ece_brier.png",
    #     f"{_exp_rel}/figures/shap_beeswarm_xgb.png",
    #     f"{_exp_rel}/figures/shap_bar_xgb.png",
    #     f"{_exp_rel}/figures/optuna_history_hgt_standalone.png",
    #     f"{_exp_rel}/figures/optuna_history_xgb_tabular.png",
    #     f"{_exp_rel}/figures/optuna_history_rf_tabular.png",
    #     f"{_exp_rel}/figures/optuna_history_stacking_meta.png",
    #     f"{_exp_rel}/figures/loss_metric_curves_hgt_demo.png",
    #     f"{_exp_rel}/figures/metric_curves_xgb_per_fold.png",
    # ]
    # subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", "dnjomouloic@gmail.com"])
    # subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", "Loic"])
    # subprocess.run(["git", "-C", REPO_DIR, "add"] + _to_add, check=True)
    # subprocess.run(["git", "-C", REPO_DIR, "commit", "-m",
    #     f"results(hgt_v2_optuna): {'smoke' if SMOKE_TEST else 'full'} run "
    #     f"Optuna HPO + figure phare + SHAP depuis Colab GPU"
    # ], cwd=REPO_DIR, check=True)
    # subprocess.run(["git", "-C", REPO_DIR, "push"], check=True)
    # print("Option B — Résultats committés et poussés vers", REPO_URL)

    print()
    print("RAPPEL : sans l'Option A (download) OU l'Option B (git push), TOUTES les sorties")
    print("(metrics.json, REPORT.md, figures/*.png, optuna_best_params.json)")
    print("sont DÉFINITIVEMENT PERDUES à la fermeture ou au timeout du runtime Colab.")